# FEATURE ENGINEERING

## Load Data

In [153]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df_train = pd.read_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/df_train_clean.csv')
df_test  = pd.read_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/df_test_clean.csv')

print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)

Train shape: (78772, 46)
Test  shape: (42422, 46)


In [154]:
display(df_train.isna().sum())
display(df_test.isna().sum())

Id                            0
match_id                      0
date                          0
gender                        0
team                          0
opponent                      0
is_home                       0
neutral                       0
tournament                    0
venue_country                 0
team_goals                    0
opp_goals                     0
days_since_last_match_team    0
days_since_last_match_opp     0
elo_team                      0
elo_opponent                  0
confederation_team            0
confederation_opp             0
population_team               0
population_opp                0
gdp_per_capita_team           0
gdp_per_capita_opp            0
altitude_venue                0
distance_travel_team          0
distance_travel_opp           0
temperature_venue             0
is_train                      0
first_match_team              0
first_match_opp               0
match_key                     0
team_points_last5             0
team_poi

Id                                0
match_id                          0
date                              0
gender                            0
team                              0
opponent                          0
is_home                           0
neutral                           0
tournament                        0
venue_country                     0
team_goals                    42422
opp_goals                     42422
days_since_last_match_team        0
days_since_last_match_opp         0
elo_team                      42422
elo_opponent                  42422
confederation_team                0
confederation_opp                 0
population_team                   0
population_opp                    0
gdp_per_capita_team               0
gdp_per_capita_opp                0
altitude_venue                    0
distance_travel_team              0
distance_travel_opp               0
temperature_venue                 0
is_train                          0
first_match_team            

## Merge & Sort

In [155]:
df = pd.concat([df_train, df_test], axis=0, ignore_index=True)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print('Combined shape:', df.shape)

Combined shape: (121194, 46)


## Tournament Tier

In [156]:
tier1 = [
    'FIFA World Cup', 'UEFA Euro', 'Copa América',
    'African Cup of Nations', 'AFC Asian Cup',
    'Gold Cup', 'OFC Nations Cup', 'CONCACAF Championship', 'Olympic Games'
]
tier2 = [
    'FIFA World Cup qualification', 'UEFA Euro qualification',
    'African Cup of Nations qualification', 'AFC Asian Cup qualification',
    'Copa América qualification', 'Gold Cup qualification',
    'CONCACAF Nations League', 'UEFA Nations League', 'AFF Championship'
]
tier3 = [
    "King's Cup", 'Algarve Cup', 'Merdeka Tournament', 'Gulf Cup',
    'CECAFA Cup', 'Nordic Championship', 'Island Games',
    'CFU Caribbean Cup', 'CFU Caribbean Cup qualification',
    'British Home Championship', 'Southeast Asian Games',
    'South Pacific Games', 'Asian Games', 'Muratti Vase',
    'Amilcar Cabral Cup', 'AFC Championship'
]

def get_tournament_tier(t):
    if t in tier1: return 1
    if t in tier2: return 2
    if t in tier3: return 3
    if 'Friendly' in str(t): return 4
    return 3

df['tournament_tier'] = df['tournament'].apply(get_tournament_tier)
print(df['tournament_tier'].value_counts())

tournament_tier
2    39870
4    39004
3    31430
1    10890
Name: count, dtype: int64


## Temporal Features

In [157]:
df['year']           = df['date'].dt.year
df['month']          = df['date'].dt.month
df['is_recent']      = (df['year'] >= 2000).astype(int)
df['is_very_recent'] = (df['year'] >= 2010).astype(int)
df['season_num']     = df['month'].map({
    12: 0, 1: 0, 2: 0,
    3: 1,  4: 1, 5: 1,
    6: 2,  7: 2, 8: 2,
    9: 3, 10: 3, 11: 3
})
print(df[['year', 'month', 'is_recent', 'is_very_recent', 'season_num']].head())

   year  month  is_recent  is_very_recent  season_num
0  1872     11          0               0           3
1  1872     11          0               0           3
2  1873      3          0               0           1
3  1873      3          0               0           1
4  1874      3          0               0           1


## ELO Features + Propagation ke Test
> Test set tidak punya ELO. Kita isi dengan ELO terakhir yang diketahui per tim dari train.

In [158]:
df.columns

Index(['Id', 'match_id', 'date', 'gender', 'team', 'opponent', 'is_home',
       'neutral', 'tournament', 'venue_country', 'team_goals', 'opp_goals',
       'days_since_last_match_team', 'days_since_last_match_opp', 'elo_team',
       'elo_opponent', 'confederation_team', 'confederation_opp',
       'population_team', 'population_opp', 'gdp_per_capita_team',
       'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team',
       'distance_travel_opp', 'temperature_venue', 'is_train',
       'first_match_team', 'first_match_opp', 'match_key', 'team_points_last5',
       'team_points_last10', 'team_gd_last5', 'team_avg_goals_last5',
       'team_avg_conceded_last5', 'team_win_rate_last10', 'h2h_points_last5',
       'h2h_gd_last5', 'opp_points_last5', 'opp_points_last10', 'opp_gd_last5',
       'opp_avg_goals_last5', 'opp_avg_conceded_last5', 'opp_win_rate_last10',
       'points_last5_diff', 'gd_last5_diff', 'tournament_tier', 'year',
       'month', 'is_recent', 'is_very_recent',

In [159]:
display(df[df['is_train'] == 1].isna().sum().sum())
display(df[df['is_train'] == 0].isna().sum().sum())
display(df.shape)

np.int64(0)

np.int64(478360)

(121194, 52)

In [160]:
# ELO dari kolom asli (train punya, test NaN)
df['elo_diff']     = df['elo_team'] - df['elo_opponent']
df['elo_ratio']    = df['elo_team'] / (df['elo_opponent'] + 1e-5)

In [161]:
display(df[df['is_train'] == 1].isna().sum().sum())
display(df[df['is_train'] == 0].isna().sum().sum())
display(df.shape)

np.int64(0)

np.int64(563204)

(121194, 54)

## Form & Momentum Features

In [162]:
df['attack_vs_def_team']      = df['team_avg_goals_last5'] - df['opp_avg_conceded_last5']
df['attack_vs_def_opp']       = df['opp_avg_goals_last5']  - df['team_avg_conceded_last5']
df['xg_advantage']            = df['attack_vs_def_team'] - df['attack_vs_def_opp']
df['momentum_diff']           = df['team_win_rate_last10'] - df['opp_win_rate_last10']
df['momentum_ratio']          = df['team_win_rate_last10'] / (df['opp_win_rate_last10'] + 1e-5)
df['pts_diff_last10']         = df['team_points_last10'] - df['opp_points_last10']
df['team_attack_consistency'] = df['team_avg_goals_last5'] / (df['team_avg_conceded_last5'] + 1e-5)
df['opp_attack_consistency']  = df['opp_avg_goals_last5']  / (df['opp_avg_conceded_last5']  + 1e-5)
df['attack_consistency_diff'] = df['team_attack_consistency'] - df['opp_attack_consistency']

In [163]:
display(df[df['is_train'] == 1].isna().sum().sum())
display(df[df['is_train'] == 0].isna().sum().sum())
display(df.shape)

np.int64(0)

np.int64(811458)

(121194, 63)

## H2H Features

In [164]:
df['h2h_dominance']    = df['h2h_points_last5'] / 15
df['h2h_gd_per_match'] = df['h2h_gd_last5'] / 5
df['h2h_missing']      = df['h2h_points_last5'].isna().astype(int)

In [165]:
display(df[df['is_train'] == 1].isna().sum().sum())
display(df[df['is_train'] == 0].isna().sum().sum())
display(df.shape)

np.int64(0)

np.int64(832110)

(121194, 66)

## Home Advantage Features

In [166]:
df['home_and_stronger']   = ((df['is_home'] == 1) & (df['elo_diff'] > 0)).astype(int)

In [167]:
display(df[df['is_train'] == 1].isna().sum().sum())
display(df[df['is_train'] == 0].isna().sum().sum())
display(df.shape)

np.int64(0)

np.int64(832110)

(121194, 67)

## Environmental Features

In [168]:
df['travel_fatigue_diff']   = df['distance_travel_team'] - df['distance_travel_opp']
df['total_travel']          = df['distance_travel_team'] + df['distance_travel_opp']
df['altitude_away_penalty'] = df['altitude_venue'] * (1 - df['is_home'])
df['gdp_diff']              = df['gdp_per_capita_team'] - df['gdp_per_capita_opp']
df['gdp_ratio']             = df['gdp_per_capita_team'] / (df['gdp_per_capita_opp'] + 1e-5)
df['population_diff']       = df['population_team'] - df['population_opp']
df['population_ratio']      = np.log1p(df['population_team']) - np.log1p(df['population_opp'])
df['temp_extreme']          = (df['temperature_venue'] - 20).abs()

In [169]:
display(df[df['is_train'] == 1].isna().sum().sum())
display(df[df['is_train'] == 0].isna().sum().sum())
display(df.shape)

np.int64(0)

np.int64(832110)

(121194, 75)

## Rest / Fatigue Features

In [170]:
df['rest_diff'] = df['days_since_last_match_team'] - df['days_since_last_match_opp']

def rest_cat(d):
    if pd.isna(d): return 1
    if d <= 4:  return 0
    if d <= 10: return 1
    return 2

df['rest_cat_team']  = df['days_since_last_match_team'].apply(rest_cat)
df['rest_cat_opp']   = df['days_since_last_match_opp'].apply(rest_cat)
df['rest_advantage'] = df['rest_cat_team'] - df['rest_cat_opp']

In [171]:
display(df[df['is_train'] == 1].isna().sum().sum())
display(df[df['is_train'] == 0].isna().sum().sum())
display(df.shape)

np.int64(0)

np.int64(832110)

(121194, 79)

In [172]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121194 entries, 0 to 121193
Data columns (total 79 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   Id                          121194 non-null  object        
 1   match_id                    121194 non-null  object        
 2   date                        121194 non-null  datetime64[ns]
 3   gender                      121194 non-null  object        
 4   team                        121194 non-null  object        
 5   opponent                    121194 non-null  object        
 6   is_home                     121194 non-null  int64         
 7   neutral                     121194 non-null  int64         
 8   tournament                  121194 non-null  object        
 9   venue_country               121194 non-null  object        
 10  team_goals                  78772 non-null   float64       
 11  opp_goals                   78772 non-n

# SPLITTING

In [173]:
df = df.sort_values(by=['team', 'opponent', 'date'])

In [174]:
df_copy = df.copy()
df_train_fe = df_copy[df_copy['is_train'] == 1]
df_test_fe = df_copy[df_copy['is_train'] == 0]

In [175]:
categorical_columns = ['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']
unnecessary_columns = ['match_key', 'is_train', 'date']

In [176]:
for dataframe in [df_train_fe, df_test_fe]:
    dataframe[categorical_columns] = dataframe[categorical_columns].astype('category')

df_train_fe = df_train_fe.drop(columns=unnecessary_columns)
df_test_fe = df_test_fe.drop(columns=unnecessary_columns)

display(df_train_fe.info())
display(df_test_fe.info())

<class 'pandas.core.frame.DataFrame'>
Index: 78772 entries, 23511 to 68362
Data columns (total 76 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   Id                          78772 non-null  object  
 1   match_id                    78772 non-null  object  
 2   gender                      78772 non-null  category
 3   team                        78772 non-null  category
 4   opponent                    78772 non-null  category
 5   is_home                     78772 non-null  int64   
 6   neutral                     78772 non-null  int64   
 7   tournament                  78772 non-null  category
 8   venue_country               78772 non-null  category
 9   team_goals                  78772 non-null  float64 
 10  opp_goals                   78772 non-null  float64 
 11  days_since_last_match_team  78772 non-null  float64 
 12  days_since_last_match_opp   78772 non-null  float64 
 13  elo_team         

None

<class 'pandas.core.frame.DataFrame'>
Index: 42422 entries, 82209 to 94652
Data columns (total 76 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   Id                          42422 non-null  object  
 1   match_id                    42422 non-null  object  
 2   gender                      42422 non-null  category
 3   team                        42422 non-null  category
 4   opponent                    42422 non-null  category
 5   is_home                     42422 non-null  int64   
 6   neutral                     42422 non-null  int64   
 7   tournament                  42422 non-null  category
 8   venue_country               42422 non-null  category
 9   team_goals                  0 non-null      float64 
 10  opp_goals                   0 non-null      float64 
 11  days_since_last_match_team  42422 non-null  float64 
 12  days_since_last_match_opp   42422 non-null  float64 
 13  elo_team         

None

In [177]:
unique_matches = df_train_fe['match_id'].unique()

match_split_index = int(len(unique_matches)*0.8)

train_match_id = unique_matches[:match_split_index]
test_match_id = unique_matches[match_split_index:]

train_df = df_train_fe[df_train_fe['match_id'].isin(train_match_id)]
test_df = df_train_fe[df_train_fe['match_id'].isin(test_match_id)]

target = ['team_goals', 'opp_goals']

X_train = train_df.drop(columns=target)
y_train = train_df[target]

X_test = test_df.drop(columns=target)
y_test = test_df[target]

display(X_train.shape)
display(X_test.shape)

(63016, 74)

(15756, 74)

In [178]:
display(X_train.columns)

Index(['Id', 'match_id', 'gender', 'team', 'opponent', 'is_home', 'neutral',
       'tournament', 'venue_country', 'days_since_last_match_team',
       'days_since_last_match_opp', 'elo_team', 'elo_opponent',
       'confederation_team', 'confederation_opp', 'population_team',
       'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp',
       'altitude_venue', 'distance_travel_team', 'distance_travel_opp',
       'temperature_venue', 'first_match_team', 'first_match_opp',
       'team_points_last5', 'team_points_last10', 'team_gd_last5',
       'team_avg_goals_last5', 'team_avg_conceded_last5',
       'team_win_rate_last10', 'h2h_points_last5', 'h2h_gd_last5',
       'opp_points_last5', 'opp_points_last10', 'opp_gd_last5',
       'opp_avg_goals_last5', 'opp_avg_conceded_last5', 'opp_win_rate_last10',
       'points_last5_diff', 'gd_last5_diff', 'tournament_tier', 'year',
       'month', 'is_recent', 'is_very_recent', 'season_num', 'elo_diff',
       'elo_ratio', 'attack_vs_de

# ENCODING

## gender

In [179]:
gender_map = {
    'M': 1,
    'W': 0
}

X_train['gender'] = X_train['gender'].map(gender_map)
X_test['gender'] = X_test['gender'].map(gender_map)

df_test_fe['gender'] = df_test_fe['gender'].map(gender_map)

### confederation_*

In [180]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_ohe = ohe.fit_transform(X_train[['confederation_team', 'confederation_opp']])
X_test_ohe = ohe.transform(X_test[['confederation_team', 'confederation_opp']])
df_test_fe_ohe = ohe.transform(df_test_fe[['confederation_team', 'confederation_opp']])


X_train_df = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(), index=X_train.index)
X_test_df = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(), index=X_test.index)
df_test_fe_df = pd.DataFrame(df_test_fe_ohe, columns=ohe.get_feature_names_out(), index=df_test_fe.index)

In [181]:
X_train = pd.concat([X_train, X_train_df], axis=1)
X_test = pd.concat([X_test, X_test_df], axis=1)
df_test_fe = pd.concat([df_test_fe, df_test_fe_df], axis=1)

X_train = X_train.drop(columns=['confederation_team', 'confederation_opp'])
X_test = X_test.drop(columns=['confederation_team', 'confederation_opp'])
df_test_fe = df_test_fe.drop(columns=['confederation_team', 'confederation_opp'])

In [182]:
display(X_train.head())
display(X_test.head())

,Id,match_id,gender,team,opponent,is_home,neutral,tournament,venue_country,days_since_last_match_team,days_since_last_match_opp,elo_team,elo_opponent,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,first_match_team,first_match_opp,team_points_last5,team_points_last10,team_gd_last5,team_avg_goals_last5,team_avg_conceded_last5,team_win_rate_last10,h2h_points_last5,h2h_gd_last5,opp_points_last5,opp_points_last10,opp_gd_last5,opp_avg_goals_last5,opp_avg_conceded_last5,opp_win_rate_last10,points_last5_diff,gd_last5_diff,tournament_tier,year,month,is_recent,is_very_recent,season_num,elo_diff,elo_ratio,attack_vs_def_team,attack_vs_def_opp,xg_advantage,momentum_diff,momentum_ratio,pts_diff_last10,team_attack_consistency,opp_attack_consistency,attack_consistency_diff,h2h_dominance,h2h_gd_per_match,h2h_missing,home_and_stronger,travel_fatigue_diff,total_travel,altitude_away_penalty,gdp_diff,gdp_ratio,population_diff,population_ratio,temp_extreme,rest_diff,rest_cat_team,rest_cat_opp,rest_advantage,confederation_team_AFC,confederation_team_CAF,confederation_team_CONCACAF,confederation_team_CONMEBOL,confederation_team_OFC,confederation_team_UEFA,confederation_team_Unknown,confederation_opp_AFC,confederation_opp_CAF,confederation_opp_CONCACAF,confederation_opp_CONMEBOL,confederation_opp_OFC,confederation_opp_UEFA,confederation_opp_Unknown
23511,M011688_Afghanistan,M011688,1,Afghanistan,Bangladesh,0,0,AFC Asian Cup qualification,Bangladesh,589.0,0.0,1340.174305,1381.026480,12486631.0,83929765.0,275.738115,200.769677,10.0,2551.703115,0.000000,25.639990,0,1,4.0,5.0,-5.0,0.6,1.6,0.1,0.0,0.0,0.0,1.0,-22.0,0.2,4.6,0.0,4.0,17.0,2,1979,3,0,0,1,-40.852175,0.970419,-4.0,-1.4,-2.6,0.1,10000.0000,4.0,0.374998,0.043478,0.331519,0.000000,0.0,0,0,2551.703115,2551.703115,10.0,74.968438,1.373405,-71443134.0,-1.905322,5.639990,589.0,2,0,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
70545,M031805_Afghanistan,M031805,1,Afghanistan,Bangladesh,0,1,AFC Challenge Cup qualification,Kyrgyzstan,192.0,0.0,1217.305507,1238.634298,28189672.0,148391139.0,364.663542,634.987070,127.0,1844.541334,0.000000,8.165958,0,1,5.0,5.0,-3.0,1.2,1.8,0.1,0.0,-1.0,1.0,2.0,-11.0,0.8,3.0,0.0,4.0,8.0,3,2008,5,1,0,1,-21.328791,0.982780,-1.8,-1.0,-0.8,0.1,10000.0000,3.0,0.666663,0.266666,0.399997,0.000000,-0.2,0,0,1844.541334,1844.541334,127.0,-270.323528,0.574285,-120201467.0,-1.660896,11.834042,192.0,2,0,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
70888,M031948_Afghanistan,M031948,1,Afghanistan,Bangladesh,0,1,SAFF Cup,Sri Lanka,2.0,0.0,1239.249260,1222.184346,28189672.0,148391139.0,364.663542,634.987070,14.5,3355.708894,2119.239639,27.673304,0,1,5.0,10.0,-3.0,0.8,1.4,0.2,1.0,-1.0,3.0,4.0,-6.0,0.6,1.8,0.0,2.0,3.0,3,2008,6,1,0,2,17.064914,1.013963,-1.0,-0.8,-0.2,0.2,20000.0000,6.0,0.571424,0.333331,0.238093,0.066667,-0.2,0,0,1236.469255,5474.948533,14.5,-270.323528,0.574285,-120201467.0,-1.660896,7.673304,2.0,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
70966,M031995_Afghanistan,M031995,1,Afghanistan,Bhutan,1,1,SAFF Cup,Sri Lanka,2.0,2.0,1239.249260,1115.375615,28189672.0,705516.0,364.663542,1828.154677,14.5,3355.708894,2473.933070,27.673304,0,0,6.0,11.0,0.0,1.2,1.2,0.2,0.0,0.0,2.0,6.0,-7.0,0.6,2.0,0.1,4.0,7.0,3,2008,6,1,0,2,123.873646,1.111060,-0.8,-0.6,-0.2,0.1,1.9998,5.0,0.999992,0.299999,0.699993,0.000000,0.0,0,1,881.775824,5829.641964,0.0,-1463.491135,0.199471,27484156.0,3.687780,7.673304,0.0,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
77773,M034582_Afghanistan,M034582,1,Afghanistan,Bhutan,0,1,AFC Challenge Cup qualification,India,95.0,0.0,1143.329449,1042.110790,28189672.0,705516.0,591.190030,2563.261224,205.5,1844.541334,1388.045688,24.293232,0,1,1.0,2.0,-20.0,0.4,4.4,0.0,0.0,-2.0,1.0,1.0,-29.0,0.4,6.2,0.0,0.0,9.0,3,2011,3,1,1,1,101.218659,1.097128,-5.8,-4.0,-1.8,0.0,0.0000,1.0,0.090909,0.064516,0.026393,0.000000,-0.4,0,0,456.4956

,Id,match_id,gender,team,opponent,is_home,neutral,tournament,venue_country,days_since_last_match_team,days_since_last_match_opp,elo_team,elo_opponent,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,first_match_team,first_match_opp,team_points_last5,team_points_last10,team_gd_last5,team_avg_goals_last5,team_avg_conceded_last5,team_win_rate_last10,h2h_points_last5,h2h_gd_last5,opp_points_last5,opp_points_last10,opp_gd_last5,opp_avg_goals_last5,opp_avg_conceded_last5,opp_win_rate_last10,points_last5_diff,gd_last5_diff,tournament_tier,year,month,is_recent,is_very_recent,season_num,elo_diff,elo_ratio,attack_vs_def_team,attack_vs_def_opp,xg_advantage,momentum_diff,momentum_ratio,pts_diff_last10,team_attack_consistency,opp_attack_consistency,attack_consistency_diff,h2h_dominance,h2h_gd_per_match,h2h_missing,home_and_stronger,travel_fatigue_diff,total_travel,altitude_away_penalty,gdp_diff,gdp_ratio,population_diff,population_ratio,temp_extreme,rest_diff,rest_cat_team,rest_cat_opp,rest_advantage,confederation_team_AFC,confederation_team_CAF,confederation_team_CONCACAF,confederation_team_CONMEBOL,confederation_team_OFC,confederation_team_UEFA,confederation_team_Unknown,confederation_opp_AFC,confederation_opp_CAF,confederation_opp_CONCACAF,confederation_opp_CONMEBOL,confederation_opp_OFC,confederation_opp_UEFA,confederation_opp_Unknown
57921,M026895_Mali,M026895,1,Mali,Niger,0,0,Friendly,Niger,16.0,0.0,1568.462790,1443.910261,11239101.0,11622665.0,336.418739,228.235909,299.5,7336.303805,0.000000,19.930996,0,1,7.0,15.0,-1.0,1.8,2.0,0.4,10.0,3.0,4.0,11.0,-10.0,1.0,3.0,0.3,3.0,9.0,4,2002,12,1,0,0,124.552529,1.086261,-1.2,-1.0,-0.2,0.1,1.333289,4.0,0.899996,0.333332,0.566663,0.666667,0.6,0,0,7336.303805,7336.303805,299.5,108.182831,1.473996,-383564.0,-0.033558,0.069004,16.0,2,0,2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
10134,M005064_Mali,M005064,1,Mali,Nigeria,1,0,Friendly,Mali,47.0,36.0,1501.067934,1537.864937,6153587.0,55569264.0,48.279925,92.960466,-9999.0,0.000000,7402.607417,32.532173,0,0,3.0,3.0,0.0,3.0,3.0,0.5,0.0,0.0,8.0,16.0,7.0,2.8,1.4,0.4,-5.0,-7.0,4,1960,6,0,0,2,-36.797003,0.976073,1.6,-0.2,1.8,0.1,1.249969,-13.0,0.999997,1.999986,-0.999989,0.000000,0.0,0,0,-7402.607417,7402.607417,-0.0,-44.680541,0.519360,-49415677.0,-2.200595,12.532173,11.0,2,2,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
10297,M005144_Mali,M005144,1,Mali,Nigeria,0,0,Friendly,Nigeria,3.0,0.0,1534.877432,1511.044511,6153587.0,55569264.0,48.279925,92.960466,376.5,7402.607417,0.000000,26.371895,0,1,9.0,9.0,-2.0,2.2,2.6,0.6,3.0,1.0,5.0,15.0,-1.0,1.8,2.0,0.4,4.0,-1.0,4,1960,10,0,0,3,23.832921,1.015772,0.2,-0.8,1.0,0.2,1.499963,-6.0,0.846151,0.899996,-0.053845,0.200000,0.2,0,0,7402.607417,7402.607417,376.5,-44.680541,0.519360,-49415677.0,-2.200595,6.371895,3.0,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
18229,M009085_Mali,M009085,1,Mali,Nigeria,1,0,Friendly,Mali,262.0,11.0,1539.079795,1640.461480,6153587.0,55569264.0,79.157411,209.226045,-9999.0,0.000000,7402.607417,24.874743,0,0,6.0,16.0,0.0,2.2,2.2,0.4,6.0,2.0,13.0,22.0,6.0,2.4,1.2,0.6,-7.0,-6.0,4,1972,11,0,0,3,-101.381685,0.938199,1.0,0.2,0.8,-0.2,0.666656,-6.0,0.999995,1.999983,-0.999988,0.400000,0.4,0,0,-7402.607417,7402.607417,-0.0,-130.068634,0.378334,-49415677.0,-2.200595,4.874743,251.0,2,2,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
18283,M009113_Mali,M009113,1,Mali,Nigeria,0,0,Friendly,Nigeria,12.0,0.0,1547.133793,1649.661401,6153587.0,55569264.0,79.157411,209.226045,376.5,7402.607417,0.000000,25.731754,0,1,8.0,16.0,1.0,2.0,1.8,0.4,9.0,3.0,12.0,23.0,4.0,2.0,1.2,0.7,-4.0,-3.0,4,1972,12,0,0,0,-102.527607,0.937849,0.8,0.2,0.6,-0.3,0.571420,-7.0,1.111105,1.666653,-0.555548,0.600000,0.6,0,0,7402.607417,7402.607417,376.5,-130.068634,0.378334,-49415677.0,-2.200595,5.731754,12.0,2,0,2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [183]:
display(df_test_fe.head())

,Id,match_id,gender,team,opponent,is_home,neutral,tournament,venue_country,team_goals,opp_goals,days_since_last_match_team,days_since_last_match_opp,elo_team,elo_opponent,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,first_match_team,first_match_opp,team_points_last5,team_points_last10,team_gd_last5,team_avg_goals_last5,team_avg_conceded_last5,team_win_rate_last10,h2h_points_last5,h2h_gd_last5,opp_points_last5,opp_points_last10,opp_gd_last5,opp_avg_goals_last5,opp_avg_conceded_last5,opp_win_rate_last10,points_last5_diff,gd_last5_diff,tournament_tier,year,month,is_recent,is_very_recent,season_num,elo_diff,elo_ratio,attack_vs_def_team,attack_vs_def_opp,xg_advantage,momentum_diff,momentum_ratio,pts_diff_last10,team_attack_consistency,opp_attack_consistency,attack_consistency_diff,h2h_dominance,h2h_gd_per_match,h2h_missing,home_and_stronger,travel_fatigue_diff,total_travel,altitude_away_penalty,gdp_diff,gdp_ratio,population_diff,population_ratio,temp_extreme,rest_diff,rest_cat_team,rest_cat_opp,rest_advantage,confederation_team_AFC,confederation_team_CAF,confederation_team_CONCACAF,confederation_team_CONMEBOL,confederation_team_OFC,confederation_team_UEFA,confederation_team_Unknown,confederation_opp_AFC,confederation_opp_CAF,confederation_opp_CONCACAF,confederation_opp_CONMEBOL,confederation_opp_OFC,confederation_opp_UEFA,confederation_opp_Unknown
82209,M036241_Abkhazia,M036241,1,Abkhazia,Artsakh,1,0,Friendly,Georgia,NaN,NaN,0.0,0.0,NaN,NaN,9893316.0,9893316.0,3784.449144,3784.449144,598.0,0.0,0.0,15.811390,1,1,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,4,2012,9,1,1,3,NaN,NaN,0.0,0.0,0.0,0.0,0.99998,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.188610,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
82499,M036386_Abkhazia,M036386,1,Abkhazia,Artsakh,0,0,Friendly,Azerbaijan,NaN,NaN,26.0,26.0,NaN,NaN,9893316.0,9893316.0,3784.449144,3784.449144,54.0,0.0,0.0,16.320108,0,0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,4,2012,10,1,1,3,NaN,NaN,NaN,NaN,NaN,0.0,0.00000,0.0,NaN,NaN,NaN,0.0,NaN,0,0,0.0,0.0,54.0,0.0,1.0,0.0,0.0,3.679892,0.0,2,2,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
99864,M042438_Abkhazia,M042438,1,Abkhazia,Artsakh,0,0,CONIFA European Football Cup,Azerbaijan,NaN,NaN,1.0,1.0,NaN,NaN,9893316.0,9893316.0,3784.449144,3784.449144,54.0,0.0,0.0,24.229780,0,0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,3,2019,6,1,1,2,NaN,NaN,NaN,NaN,NaN,0.0,0.00000,0.0,NaN,NaN,NaN,0.0,NaN,0,0,0.0,0.0,54.0,0.0,1.0,0.0,0.0,4.229780,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
91625,M039667_Abkhazia,M039667,1,Abkhazia,Chagos Islands,1,0,CONIFA World Football Cup,Georgia,NaN,NaN,8.0,161.0,NaN,NaN,9893316.0,9893316.0,3784.449144,3784.449144,598.0,0.0,0.0,12.219041,0,0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,3,2016,5,1,1,1,NaN,NaN,NaN,NaN,NaN,0.0,0.00000,0.0,NaN,NaN,NaN,NaN,NaN,1,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,7.780959,-153.0,1,2,-1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
99840,M042417_Abkhazia,M042417,1,Abkhazia,Chameria,1,1,CONIFA European Football Cup,Azerbaijan,NaN,NaN,358.0,0.0,NaN,NaN,9893316.0,9893316.0,3784.449144,3784.449144,54.0,0.0,0.0,24.229780,0,1,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,NaN,3,2019,6,1,1,2,NaN,NaN,NaN,NaN,NaN,-0.5,0.00000,0.0,NaN,0.0,NaN,0.0,0.0,0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.229780,358.0,2,0,2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [184]:
display(X_train.shape)
display(X_test.shape)
display(df_test_fe.shape)

(63016, 86)

(15756, 86)

(42422, 88)

In [185]:
display(X_train.columns.to_list())
display(df_test_fe.columns.to_list())

['Id',
 'match_id',
 'gender',
 'team',
 'opponent',
 'is_home',
 'neutral',
 'tournament',
 'venue_country',
 'days_since_last_match_team',
 'days_since_last_match_opp',
 'elo_team',
 'elo_opponent',
 'population_team',
 'population_opp',
 'gdp_per_capita_team',
 'gdp_per_capita_opp',
 'altitude_venue',
 'distance_travel_team',
 'distance_travel_opp',
 'temperature_venue',
 'first_match_team',
 'first_match_opp',
 'team_points_last5',
 'team_points_last10',
 'team_gd_last5',
 'team_avg_goals_last5',
 'team_avg_conceded_last5',
 'team_win_rate_last10',
 'h2h_points_last5',
 'h2h_gd_last5',
 'opp_points_last5',
 'opp_points_last10',
 'opp_gd_last5',
 'opp_avg_goals_last5',
 'opp_avg_conceded_last5',
 'opp_win_rate_last10',
 'points_last5_diff',
 'gd_last5_diff',
 'tournament_tier',
 'year',
 'month',
 'is_recent',
 'is_very_recent',
 'season_num',
 'elo_diff',
 'elo_ratio',
 'attack_vs_def_team',
 'attack_vs_def_opp',
 'xg_advantage',
 'momentum_diff',
 'momentum_ratio',
 'pts_diff_last

['Id',
 'match_id',
 'gender',
 'team',
 'opponent',
 'is_home',
 'neutral',
 'tournament',
 'venue_country',
 'team_goals',
 'opp_goals',
 'days_since_last_match_team',
 'days_since_last_match_opp',
 'elo_team',
 'elo_opponent',
 'population_team',
 'population_opp',
 'gdp_per_capita_team',
 'gdp_per_capita_opp',
 'altitude_venue',
 'distance_travel_team',
 'distance_travel_opp',
 'temperature_venue',
 'first_match_team',
 'first_match_opp',
 'team_points_last5',
 'team_points_last10',
 'team_gd_last5',
 'team_avg_goals_last5',
 'team_avg_conceded_last5',
 'team_win_rate_last10',
 'h2h_points_last5',
 'h2h_gd_last5',
 'opp_points_last5',
 'opp_points_last10',
 'opp_gd_last5',
 'opp_avg_goals_last5',
 'opp_avg_conceded_last5',
 'opp_win_rate_last10',
 'points_last5_diff',
 'gd_last5_diff',
 'tournament_tier',
 'year',
 'month',
 'is_recent',
 'is_very_recent',
 'season_num',
 'elo_diff',
 'elo_ratio',
 'attack_vs_def_team',
 'attack_vs_def_opp',
 'xg_advantage',
 'momentum_diff',
 'mom

### venue_country & tournament

In [186]:
freq_cols = ['venue_country', 'tournament']

for col in freq_cols:
    freq_map = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq_map)
    X_test[col] = X_test[col].map(freq_map)
    X_test[col] = X_test[col].fillna(0)
    df_test_fe[col] = df_test_fe[col].map(freq_map)
    df_test_fe[col] = df_test_fe[col].fillna(0)

In [187]:
display(X_train.shape)
display(X_test.shape)
display(df_test_fe.shape)

(63016, 86)

(15756, 86)

(42422, 88)

In [188]:
display(X_train.info())
display(X_test.info())
display(df_test_fe.info())

<class 'pandas.core.frame.DataFrame'>
Index: 63016 entries, 23511 to 78478
Data columns (total 86 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   Id                           63016 non-null  object  
 1   match_id                     63016 non-null  object  
 2   gender                       63016 non-null  category
 3   team                         63016 non-null  category
 4   opponent                     63016 non-null  category
 5   is_home                      63016 non-null  int64   
 6   neutral                      63016 non-null  int64   
 7   tournament                   63016 non-null  float64 
 8   venue_country                63016 non-null  float64 
 9   days_since_last_match_team   63016 non-null  float64 
 10  days_since_last_match_opp    63016 non-null  float64 
 11  elo_team                     63016 non-null  float64 
 12  elo_opponent                 63016 non-null  float64 
 13  po

None

<class 'pandas.core.frame.DataFrame'>
Index: 15756 entries, 57921 to 68362
Data columns (total 86 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   Id                           15756 non-null  object  
 1   match_id                     15756 non-null  object  
 2   gender                       15756 non-null  category
 3   team                         15756 non-null  category
 4   opponent                     15756 non-null  category
 5   is_home                      15756 non-null  int64   
 6   neutral                      15756 non-null  int64   
 7   tournament                   15756 non-null  float64 
 8   venue_country                15756 non-null  float64 
 9   days_since_last_match_team   15756 non-null  float64 
 10  days_since_last_match_opp    15756 non-null  float64 
 11  elo_team                     15756 non-null  float64 
 12  elo_opponent                 15756 non-null  float64 
 13  po

None

<class 'pandas.core.frame.DataFrame'>
Index: 42422 entries, 82209 to 94652
Data columns (total 88 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   Id                           42422 non-null  object  
 1   match_id                     42422 non-null  object  
 2   gender                       42422 non-null  category
 3   team                         42422 non-null  category
 4   opponent                     42422 non-null  category
 5   is_home                      42422 non-null  int64   
 6   neutral                      42422 non-null  int64   
 7   tournament                   42422 non-null  float64 
 8   venue_country                42422 non-null  float64 
 9   team_goals                   0 non-null      float64 
 10  opp_goals                    0 non-null      float64 
 11  days_since_last_match_team   42422 non-null  float64 
 12  days_since_last_match_opp    42422 non-null  float64 
 13  el

None

In [189]:
display(X_train.isna().sum())
display(X_test.isna().sum())
display(df_test_fe.isna().sum())

Id                             0
match_id                       0
gender                         0
team                           0
opponent                       0
is_home                        0
neutral                        0
tournament                     0
venue_country                  0
days_since_last_match_team     0
days_since_last_match_opp      0
elo_team                       0
elo_opponent                   0
population_team                0
population_opp                 0
gdp_per_capita_team            0
gdp_per_capita_opp             0
altitude_venue                 0
distance_travel_team           0
distance_travel_opp            0
temperature_venue              0
first_match_team               0
first_match_opp                0
team_points_last5              0
team_points_last10             0
team_gd_last5                  0
team_avg_goals_last5           0
team_avg_conceded_last5        0
team_win_rate_last10           0
h2h_points_last5               0
h2h_gd_las

Id                             0
match_id                       0
gender                         0
team                           0
opponent                       0
is_home                        0
neutral                        0
tournament                     0
venue_country                  0
days_since_last_match_team     0
days_since_last_match_opp      0
elo_team                       0
elo_opponent                   0
population_team                0
population_opp                 0
gdp_per_capita_team            0
gdp_per_capita_opp             0
altitude_venue                 0
distance_travel_team           0
distance_travel_opp            0
temperature_venue              0
first_match_team               0
first_match_opp                0
team_points_last5              0
team_points_last10             0
team_gd_last5                  0
team_avg_goals_last5           0
team_avg_conceded_last5        0
team_win_rate_last10           0
h2h_points_last5               0
h2h_gd_las

Id                                 0
match_id                           0
gender                             0
team                               0
opponent                           0
is_home                            0
neutral                            0
tournament                         0
venue_country                      0
team_goals                     42422
opp_goals                      42422
days_since_last_match_team         0
days_since_last_match_opp          0
elo_team                       42422
elo_opponent                   42422
population_team                    0
population_opp                     0
gdp_per_capita_team                0
gdp_per_capita_opp                 0
altitude_venue                     0
distance_travel_team               0
distance_travel_opp                0
temperature_venue                  0
first_match_team                   0
first_match_opp                    0
team_points_last5                  0
team_points_last10                 0
t

# EXPORT DATASETS

In [190]:
X_train.to_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/X_train.csv', index=False)
X_test.to_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/X_test.csv', index=False)

y_train.to_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/y_train.csv', index=False)
y_test.to_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/y_test.csv', index=False)

df_test_fe.to_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/df_test_fe.csv', index=False)